Работу выполнил: Резников Иван, P3122

# Лабораторная работа 8. Скрапинг

##  Цель работы
Освоение методов скрапинга данных из веб-страниц.

## Теоретические сведения
Основные понятия:

Скрейпинг / Веб-скрейпинг — это технология получения веб-данных путём извлечения их со страниц веб-ресурсов.


Объект для скрапинга: https://news.itmo.ru/ru.

Необходимо сохранить полученные данных в формате csv внутри колаба.

Структура данных :

1. Идентификатор новости (целочисленное число из URL)
2. Название новости
3. Дата её размещения
4. URL на страницу с конкретной новостью.

In [35]:
!pip install requests beautifulsoup4

In [36]:
import csv
import os
import re
import time
import random
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

DOMAIN = 'https://news.itmo.ru'
MAIN_NEWS_URL = DOMAIN + '/ru/main_news/{page}/'


## Вспомогательные функции


In [37]:
def get_soup(url):
  """Скачивает страницу и возвращает BeautifulSoup."""
  response = requests.get(url, timeout=10)
  response.raise_for_status()
  return BeautifulSoup(response.text, 'html.parser')


def extract_news_id(url):
  """Достает id у ссылки."""
  match = re.search(r'/news/(\d+)/?', url)
  return int(match.group(1)) if match else None


def extract_text(soup):
  """Достает текст статьи, игнорируя скрипты."""
  container = soup.select_one('.post-content') or soup.select_one('.content.js-mediator-article')
  if not container:
    return ''

  for bad in container.select('script, style, .ya-share2, .slick-slider'):
    bad.decompose()

  parts = []
  for tag in container.find_all(['p', 'h2', 'h3', 'h4', \
                                 'li', 'blockquote', 'figcaption']):
    text = tag.get_text(' ', strip=True)
    if text:
      parts.append(text)

  return '\n\n'.join(parts)


## Сбор общих данных

In [38]:
def collect_news_list(max_pages=5):
  """Обходит страницы main_news и собирает список новостей."""
  news_by_id = {}
  page = 1

  while True:
    print(f'Страница {page}...')
    soup = get_soup(MAIN_NEWS_URL.format(page=page))
    items = soup.select('li:has(h4 > a[href*="/news/"])')

    if not items:
      break

    for li in items:
      link = li.select_one('h4 > a')
      url = urljoin(DOMAIN, link['href'])
      news_id = extract_news_id(url)
      time_tag = li.find('time')
      date = time_tag.get_text(strip=True) if time_tag else ''
      news_by_id[news_id] = {
        'Идентификатор новости': news_id,
        'Название новости': link.get_text(strip=True),
        'Дата её размещения': date,
        'URL на страницу с конкретной новостью': url,
      }
    pagination_links = soup.select('.pagination li a')
    next_link = pagination_links[-1]['href'] if pagination_links else '#'
    if next_link == '#':
      break
    page += 1
    if max_pages and page > max_pages:
        break
    time.sleep(random.uniform(0.3, 1.0))
  return list(news_by_id.values())
news_list = collect_news_list(max_pages=None)
print(f'Собрано новостей: {len(news_list)}')
news_list[:2]

Страница 1...
Страница 2...
Страница 3...
Страница 4...
Страница 5...
Страница 6...
Страница 7...
Страница 8...
Страница 9...
Страница 10...
Страница 11...
Страница 12...
Страница 13...
Страница 14...
Страница 15...
Страница 16...
Страница 17...
Страница 18...
Страница 19...
Страница 20...
Страница 21...
Страница 22...
Страница 23...
Страница 24...
Страница 25...
Страница 26...
Страница 27...
Страница 28...
Страница 29...
Страница 30...
Страница 31...
Страница 32...
Страница 33...
Страница 34...
Страница 35...
Страница 36...
Страница 37...
Страница 38...
Страница 39...
Страница 40...
Страница 41...
Страница 42...
Страница 43...
Страница 44...
Страница 45...
Страница 46...
Страница 47...
Страница 48...
Страница 49...
Страница 50...
Страница 51...
Страница 52...
Страница 53...
Страница 54...
Страница 55...
Страница 56...
Страница 57...
Страница 58...
Страница 59...
Страница 60...
Страница 61...
Страница 62...
Страница 63...
Страница 64...
Страница 65...
Страница 66...
Страница 67...
Стра

[{'Идентификатор новости': 14850,
  'Название новости': 'В России откроют центр компетенций для усовершенствования дата-центров',
  'Дата её размещения': '20 Мая 2026',
  'URL на страницу с конкретной новостью': 'https://news.itmo.ru/ru/science/photonics/news/14850/'},
 {'Идентификатор новости': 14258,
  'Название новости': 'Яндекс, Сбер, VK и не только: магистратуры ИТМО, которые поддерживают крупные компании',
  'Дата её размещения': '14 Апреля 2026',
  'URL на страницу с конкретной новостью': 'https://news.itmo.ru/ru/education/trend/news/14258/'}]

## Сохранение в csv файл

In [39]:
output_csv = 'scraped_news_data.csv'

fieldnames = [
  'Идентификатор новости',
  'Название новости',
  'Дата её размещения',
  'URL на страницу с конкретной новостью',
]

with open(output_csv, 'w', newline='', encoding='utf-8') as f:
  writer = csv.DictWriter(f, fieldnames=fieldnames)
  writer.writeheader()
  writer.writerows(news_list)

print(f'Сохранено в {output_csv}')

Сохранено в scraped_news_data.csv


## Парсинг содержимого каждой новости

In [44]:
def parse_news_details(url):
  """Заходит на страницу одной новости и достаёт всё нужное."""
  soup = get_soup(url)

  news_id = extract_news_id(url)

  # дата
  time_tag = soup.select_one('article time') or \
             soup.select_one('.news-info-wrapper time')
  date = time_tag['datetime'][:10] if time_tag and time_tag.get('datetime') else ''

  # просмотры
  views_tag = soup.select_one('span.icon.eye')
  views = int(views_tag.get_text(strip=True)) if views_tag else 0

  # заголовок на самой странице
  title_tag = soup.find('h1')
  title = title_tag.get_text(strip=True) if title_tag else ''

  # теги
  tags = [tag['content'] for tag in soup.find_all('meta', property='article:tag')]

  # текст
  text = extract_text(soup)

  return {
    'Идентификатор': news_id,
    'Название новости': title,
    'Дата размещения': date,
    'Количество просмотров': views,
    'Текст': text,
    'Теги': '; '.join(tags),
  }

<Response [200]>

In [41]:
test = parse_news_details(news_list[30]['URL на страницу с конкретной новостью'])
print(f'Название: {test["Название новости"]}')
print(f'Просмотры: {test["Количество просмотров"]}')
print(f'Теги: {test["Теги"]}')
print(test['Текст'][:300])

Название: Все складывается в ИТМО: в Первом неклассическом стартовала приемная кампания
Просмотры: 34677
Теги: Правила приема абитуриентов; Магистратура; Бакалавриат; Поступление; Аспирантура; Главное; Абитуриенты
Все складывается в ИТМО

Куда поступать

Как подать заявку на поступление

Главные даты

Где следить за ходом приема

Слоган новой приемной кампании 2025 — «Все складывается в ИТМО». В Первом неклассическом могут сочетаться совершенно разные вещи: искусство и программирование, физика и киберспорт, с


In [42]:
news_list[30]

{'Идентификатор новости': 14366,
 'Название новости': 'Все складывается в ИТМО: в Первом неклассическом стартовала приемная кампания',
 'Дата её размещения': '20 Июня 2025',
 'URL на страницу с конкретной новостью': 'https://news.itmo.ru/ru/education/official/news/14366/'}

## Запустим на нашем датасете:

In [43]:
os.makedirs('news_content', exist_ok=True)

details = []

for i, item in enumerate(news_list, start=1):
  url = item['URL на страницу с конкретной новостью']
  print(f'{i}/{len(news_list)}: {url}')

  try:
    row = parse_news_details(url)
    details.append(row)
  except Exception as e:
    print(f'Ошибка для {url}: {e}')
  time.sleep(random.uniform(0.3, 1.0))

details_csv = 'news_content/news_details.csv'
detail_fields = [
  'Идентификатор',
  'Название новости',
  'Дата размещения',
  'Количество просмотров',
  'Текст',
  'Теги',
]

with open(details_csv, 'w', newline='', encoding='utf-8') as f:
  writer = csv.DictWriter(f, fieldnames=detail_fields, quoting=csv.QUOTE_ALL)
  writer.writeheader()
  writer.writerows(details)

print(f'Сохранено в {details_csv}')

1/1131: https://news.itmo.ru/ru/science/photonics/news/14850/
2/1131: https://news.itmo.ru/ru/education/trend/news/14258/
3/1131: https://news.itmo.ru/ru/education/students/news/14779/
4/1131: https://news.itmo.ru/ru/university_live/achievements/news/14727/
5/1131: https://news.itmo.ru/ru/university_live/achievements/news/14716/
6/1131: https://news.itmo.ru/ru/education/trend/news/14692/
7/1131: https://news.itmo.ru/ru/university_live/achievements/news/14670/
8/1131: https://news.itmo.ru/ru/university_live/achievements/news/14646/
9/1131: https://news.itmo.ru/ru/education/official/news/14601/
10/1131: https://news.itmo.ru/ru/education/trend/news/14591/
11/1131: https://news.itmo.ru/ru/university_live/achievements/news/14590/
12/1131: https://news.itmo.ru/ru/startups_and_business/innovations/news/14532/
13/1131: https://news.itmo.ru/ru/education/cooperation/news/14511/
14/1131: https://news.itmo.ru/ru/university_live/leisure/news/14505/
15/1131: https://news.itmo.ru/ru/university_live/l

In [45]:
import pandas as pd

summary = pd.read_csv('scraped_news_data.csv')
details = pd.read_csv('news_content/news_details.csv')

print('Общий список:', summary.shape)
print('Детали:', details.shape)
print('Пустой текст:', details['Текст'].str.strip().eq('').sum())
print('Без тегов:', details['Теги'].str.strip().eq('').sum())
summary.head(3)

Общий список: (1131, 4)
Детали: (1131, 6)
Пустой текст: 0
Без тегов: 0


,Идентификатор новости,Название новости,Дата её размещения,URL на страницу с конкретной новостью
0,14850,В России откроют центр компетенций для усоверш...,20 Мая 2026,https://news.itmo.ru/ru/science/photonics/news...
1,14258,"Яндекс, Сбер, VK и не только: магистратуры ИТМ...",14 Апреля 2026,https://news.itmo.ru/ru/education/trend/news/1...
2,14779,Миллион уже на первом курсе: ИТМО повысил стип...,26 Марта 2026,https://news.itmo.ru/ru/education/students/new...


## Выводы

В работе реализован скрапинг новостей с сайта news.itmo.ru.
Сначала с раздела main_news собран общий список новостей и сохранён в в csv файл (`scraped_news_data.csv`).
Затем для каждой новости извлечены просмотры, текст и теги; данные сохранены в `news_content/news_details.csv`.

Были использованы `requests`, `BeautifulSoup`, работа с пагинацией, CSS-селекторами и CSV.
Показано, что при парсинге важно учитывать особенности вёрстки и не смешивать разные поля (например, дату и просмотры)